In [0]:
customer_df = spark.table(
    "retail_medallion_ws.default.bronze_customer_info"
)

display(customer_df)

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
11005,AW00011005,Julio,Ruiz,S,M,2025-10-06
11006,AW00011006,Janet,Alvarez,S,F,2025-10-06
11007,AW00011007,Marco,Mehta,M,M,2025-10-06
11008,AW00011008,Rob,Verhoff,S,F,2025-10-06
11009,AW00011009,Shannon,Carlson,S,M,2025-10-06


In [0]:
customer_df.printSchema()

root
 |-- cst_id: integer (nullable = true)
 |-- cst_key: string (nullable = true)
 |-- cst_firstname: string (nullable = true)
 |-- cst_lastname: string (nullable = true)
 |-- cst_marital_status: string (nullable = true)
 |-- cst_gndr: string (nullable = true)
 |-- cst_create_date: date (nullable = true)



In [0]:
display(customer_df.limit(10))

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
11005,AW00011005,Julio,Ruiz,S,M,2025-10-06
11006,AW00011006,Janet,Alvarez,S,F,2025-10-06
11007,AW00011007,Marco,Mehta,M,M,2025-10-06
11008,AW00011008,Rob,Verhoff,S,F,2025-10-06
11009,AW00011009,Shannon,Carlson,S,M,2025-10-06


In [0]:
print("Total rows:", customer_df.count())
print("Distinct rows:", customer_df.dropDuplicates().count())

Total rows: 18494
Distinct rows: 18494


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN cst_id IS NULL THEN 1 ELSE 0 END) AS null_cst_id,
    SUM(CASE WHEN cst_key IS NULL THEN 1 ELSE 0 END) AS null_cst_key,
    SUM(CASE WHEN cst_firstname IS NULL THEN 1 ELSE 0 END) AS null_firstname,
    SUM(CASE WHEN cst_lastname IS NULL THEN 1 ELSE 0 END) AS null_lastname,
    SUM(CASE WHEN cst_gndr IS NULL THEN 1 ELSE 0 END) AS null_gender,
    SUM(CASE WHEN cst_create_date IS NULL THEN 1 ELSE 0 END) AS null_create_date
FROM retail_medallion_ws.default.bronze_customer_info;

total_rows,null_cst_id,null_cst_key,null_firstname,null_lastname,null_gender,null_create_date
18494,4,0,8,7,4578,4


In [0]:
%sql

SELECT
    cst_id,
    COUNT(*) AS record_count
FROM retail_medallion_ws.default.bronze_customer_info
WHERE cst_id IS NOT NULL
GROUP BY cst_id
HAVING COUNT(*) > 1
ORDER BY record_count DESC;

cst_id,record_count
29466,3
29473,2
29433,2
29449,2
29483,2


In [0]:
%sql

SELECT *
FROM retail_medallion_ws.default.bronze_customer_info
WHERE cst_id IS NOT NULL;

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
11000,AW00011000,Jon,Yang,M,M,2025-10-06
11001,AW00011001,Eugene,Huang,S,M,2025-10-06
11002,AW00011002,Ruben,Torres,M,M,2025-10-06
11003,AW00011003,Christy,Zhu,S,F,2025-10-06
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06
11005,AW00011005,Julio,Ruiz,S,M,2025-10-06
11006,AW00011006,Janet,Alvarez,S,F,2025-10-06
11007,AW00011007,Marco,Mehta,M,M,2025-10-06
11008,AW00011008,Rob,Verhoff,S,F,2025-10-06
11009,AW00011009,Shannon,Carlson,S,M,2025-10-06


In [0]:
%sql

SELECT *
FROM retail_medallion_ws.default.bronze_customer_info
WHERE cst_create_date IS NULL;  

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date
null,SF566,null,null,null,null,null
null,PO25,null,null,null,null,null
null,13451235,null,null,null,null,null
null,A01Ass,null,null,null,null,null


In [0]:
%sql

SELECT *
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY cst_id
            ORDER BY cst_create_date DESC NULLS LAST
        ) AS rn
    FROM retail_medallion_ws.default.bronze_customer_info
    WHERE cst_id IS NOT NULL
)
WHERE rn = 1
ORDER BY cst_id;

cst_id,cst_key,cst_firstname,cst_lastname,cst_marital_status,cst_gndr,cst_create_date,rn
11000,AW00011000,Jon,Yang,M,M,2025-10-06,1
11001,AW00011001,Eugene,Huang,S,M,2025-10-06,1
11002,AW00011002,Ruben,Torres,M,M,2025-10-06,1
11003,AW00011003,Christy,Zhu,S,F,2025-10-06,1
11004,AW00011004,Elizabeth,Johnson,S,F,2025-10-06,1
11005,AW00011005,Julio,Ruiz,S,M,2025-10-06,1
11006,AW00011006,Janet,Alvarez,S,F,2025-10-06,1
11007,AW00011007,Marco,Mehta,M,M,2025-10-06,1
11008,AW00011008,Rob,Verhoff,S,F,2025-10-06,1
11009,AW00011009,Shannon,Carlson,S,M,2025-10-06,1


In [0]:
%sql

CREATE OR REPLACE TABLE retail_medallion_ws.default.silver_customer AS

SELECT
    cst_id,
    cst_key,
    cst_firstname,
    cst_lastname,
    cst_marital_status,
    cst_gndr,
    cst_create_date
FROM (
    SELECT
        *,
        ROW_NUMBER() OVER (
            PARTITION BY cst_id
            ORDER BY cst_create_date DESC NULLS LAST
        ) AS rn
    FROM retail_medallion_ws.default.bronze_customer_info
    WHERE cst_id IS NOT NULL
)
WHERE rn = 1;

num_affected_rows,num_inserted_rows


In [0]:
%sql

SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT cst_id) AS unique_customer_ids
FROM retail_medallion_ws.default.silver_customer;

total_rows,unique_customer_ids
18484,18484
